# Anomaly Explainer — Demo Walkthrough

This notebook demonstrates the full end-to-end flow of the Anomaly Explainer agent:

1. Train an **IsolationForest** on synthetic credit-card-like features
2. Save the model as a `.pkl` file
3. **Upload** it to the `/api/upload-model` endpoint
4. **Explain** a normal transaction
5. **Explain** an anomalous transaction
6. Display SHAP / LIME / PDP plots and the LLM summary inline

**Prerequisites:** Backend must be running — `make dev-backend` from the project root.

## 0. Imports & Config

In [ ]:
import io
import json
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from IPython.display import Image, display
from sklearn.ensemble import IsolationForest

BASE_URL = "http://localhost:8000"

## 1. Generate Synthetic Data & Train IsolationForest

In [ ]:
rng = np.random.default_rng(42)

# Simulate 1 000 normal credit-card transactions
n_normal = 1000
amount   = rng.lognormal(mean=4.0, sigma=1.2, size=n_normal)   # £0 – £10 000
duration = rng.exponential(scale=60, size=n_normal)             # seconds
n_tx_24h = rng.poisson(lam=5, size=n_normal)                    # transactions/day
dist_km  = rng.exponential(scale=20, size=n_normal)             # distance from home

X_train = np.column_stack([amount, duration, n_tx_24h, dist_km])
feature_names = ["amount", "duration_s", "n_tx_24h", "dist_km"]

df_train = pd.DataFrame(X_train, columns=feature_names)
print(f"Training data: {df_train.shape}")
df_train.describe().round(2)

In [ ]:
model = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
model.fit(X_train)
print("Model trained:", model)

## 2. Save Model & Define Schema

In [ ]:
model_path = "/tmp/credit_card_if.pkl"
with open(model_path, "wb") as f:
    pickle.dump(model, f)
print(f"Model saved to {model_path}")

In [ ]:
schema = {
    "features": {
        "amount":    {"type": "float", "min": 0.0,  "max": 10000.0},
        "duration_s":{"type": "float", "min": 0.0,  "max": 3600.0},
        "n_tx_24h":  {"type": "int",   "min": 0,    "max": 50},
        "dist_km":   {"type": "float", "min": 0.0,  "max": 500.0},
    }
}
print(json.dumps(schema, indent=2))

## 3. Upload Model to API

In [ ]:
# Also send the training data as a reference CSV for better SHAP/LIME results
ref_csv_bytes = df_train.to_csv(index=False).encode()

with open(model_path, "rb") as f:
    resp = requests.post(
        f"{BASE_URL}/api/upload-model",
        data={"schema": json.dumps(schema)},
        files={
            "model_file":    ("model.pkl", f, "application/octet-stream"),
            "reference_csv": ("train.csv", ref_csv_bytes, "text/csv"),
        },
    )

resp.raise_for_status()
upload_result = resp.json()
session_id = upload_result["session_id"]
print(f"session_id : {session_id}")
print(f"model_type : {upload_result['model_type']}")
print(f"features   : {upload_result['feature_count']}")

## 4. Explain a Normal Transaction

In [ ]:
normal_record = {
    "amount":    55.0,   # typical grocery purchase
    "duration_s": 30.0,  # quick tap-and-go
    "n_tx_24h":  3,      # a few transactions today
    "dist_km":   1.2,    # near home
}

resp_normal = requests.post(
    f"{BASE_URL}/api/explain",
    json={"session_id": session_id, "input_record": normal_record},
)
resp_normal.raise_for_status()
normal_result = resp_normal.json()

pred = normal_result["prediction"]
print(f"Label       : {pred['label']}")
print(f"Score       : {pred['anomaly_score']:.4f}")
print(f"Model type  : {pred['model_type']}")
print()
print("Summary:", normal_result["summary"]["text"])

In [ ]:
def show_plots(result):
    plots = result["explanations"]["plots"]
    urls = [
        ("SHAP Waterfall",   plots.get("shap_plot_url")),
        ("SHAP Bar Chart",   plots.get("shap_force_plot_url")),
        ("LIME Bar Chart",   plots.get("lime_plot_url")),
        *[(f"PDP {i+1}", u) for i, u in enumerate(plots.get("pdp_plot_urls", []))],
    ]
    for title, url in urls:
        if url:
            print(f"--- {title} ---")
            r = requests.get(f"{BASE_URL}{url}")
            r.raise_for_status()
            display(Image(data=r.content, format="png"))

show_plots(normal_result)

## 5. Explain an Anomalous Transaction

In [ ]:
anomalous_record = {
    "amount":    9500.0,  # unusually large
    "duration_s": 5.0,    # suspiciously fast
    "n_tx_24h":  35,      # many transactions in 24 h
    "dist_km":   450.0,   # very far from home
}

resp_anom = requests.post(
    f"{BASE_URL}/api/explain",
    json={"session_id": session_id, "input_record": anomalous_record},
)
resp_anom.raise_for_status()
anom_result = resp_anom.json()

pred = anom_result["prediction"]
print(f"Label       : {pred['label']}")
print(f"Score       : {pred['anomaly_score']:.4f}")
print()
print("Summary:", anom_result["summary"]["text"])

In [ ]:
show_plots(anom_result)

## 6. Feature Contributions Table

In [ ]:
shap_vals = anom_result["explanations"]["shap_values"] or {}
lime_vals = anom_result["explanations"]["lime_weights"] or {}

all_features = sorted(
    set(shap_vals.keys()) | set(lime_vals.keys()),
    key=lambda f: abs(shap_vals.get(f, 0)),
    reverse=True,
)

rows = [
    {
        "Feature": f,
        "SHAP": round(shap_vals.get(f, float("nan")), 5),
        "LIME": round(lime_vals.get(f, float("nan")), 5),
    }
    for f in all_features
]
pd.DataFrame(rows)

## 7. Side-by-Side Score Comparison

In [ ]:
labels = ["Normal transaction", "Anomalous transaction"]
scores = [
    normal_result["prediction"]["anomaly_score"],
    anom_result["prediction"]["anomaly_score"],
]
colors = [
    "steelblue" if normal_result["prediction"]["label"] == "normal" else "tomato",
    "steelblue" if anom_result["prediction"]["label"]  == "normal" else "tomato",
]

fig, ax = plt.subplots(figsize=(6, 3))
bars = ax.barh(labels, scores, color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Anomaly score")
ax.set_title("Score comparison (negative = more anomalous for IsolationForest)")
for bar, score in zip(bars, scores):
    ax.text(score + 0.001, bar.get_y() + bar.get_height() / 2,
            f"{score:.4f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

---
## Done!

You have successfully:
- Trained and uploaded an `IsolationForest` model
- Explained both a **normal** and an **anomalous** transaction
- Inspected SHAP, LIME, and PDP explanations
- Viewed the LLM-generated (or rule-based) summary

Next steps:
- Launch the React UI (`make dev-frontend`) for an interactive experience
- Try with a **One-Class SVM** or **LOF** model
- Set `ANTHROPIC_API_KEY` in your `.env` to enable Claude-powered summaries